# 🩺 MedReason
## Structured Diagnostic Reasoning Sandbox — Powered by MedGemma (Google HAI-DEF)

**Purpose:** An interactive educational tool that helps medical students practice structured clinical reasoning using MedGemma, Google's medically-aligned language model.

**How it works:**
1. You input a patient case (symptoms, demographics, history)
2. MedGemma reasons through it using a constrained diagnostic framework
3. Output is parsed into structured fields: summary, red flags, ranked differentials, next tests, confidence

> ⚠️ **Disclaimer:** This tool is strictly for medical education and training. It does not constitute clinical advice, diagnosis, or treatment recommendations.

In [11]:
!pip install transformers torch accelerate -q
print("✅ Done")

✅ Done


In [12]:
from google.colab import userdata
import os
os.environ["HFTOKEN"] = userdata.get("HFTOKEN")
print("✅ Token loaded")

✅ Token loaded


In [13]:
from huggingface_hub import InferenceClient
import json, re, os

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HFTOKEN"]
)

MODEL_ID = "google/medgemma-4b-it"

SYSTEM_PROMPT = """You are MedReason, a structured clinical reasoning assistant built for medical education.
Your role: help medical students practice step-by-step diagnostic reasoning.
You do NOT prescribe treatment. You do NOT make final diagnoses.

Respond ONLY with this JSON — no text outside it:

{
  "summary": "2-3 sentence clinical summary",
  "risk_flags": ["flag 1", "flag 2"],
  "differential_ranked": [
    {"rank": 1, "diagnosis": "Name", "justification": "Why", "likelihood": "High"},
    {"rank": 2, "diagnosis": "Name", "justification": "Why", "likelihood": "Medium"},
    {"rank": 3, "diagnosis": "Name", "justification": "Why", "likelihood": "Low"}
  ],
  "next_tests": ["Test — reason", "Test — reason", "Test — reason"],
  "confidence_level": "High",
  "confidence_rationale": "Why this confidence level"
}"""


def build_case(age, sex, symptoms, history, vitals='', labs=''):
    text = f"PATIENT CASE\nAge: {age}\nSex: {sex}\nSymptoms: {symptoms}\nHistory: {history}"
    if vitals: text += f"\nVitals: {vitals}"
    if labs:   text += f"\nLabs: {labs}"
    text += "\n\nProvide structured diagnostic reasoning."
    return text


def query_medgemma(case_prompt):
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": case_prompt}
        ],
        max_tokens=1024,
        temperature=0.3
    )
    return response.choices[0].message.content


def parse_output(raw):
    cleaned = re.sub(r'```json|```', '', raw).strip()
    try: return json.loads(cleaned)
    except:
        m = re.search(r'\{.*\}', cleaned, re.DOTALL)
        if m:
            try: return json.loads(m.group())
            except: pass
        return {'error': 'Parse failed', 'raw': raw}


def display(parsed):
    if 'error' in parsed:
        print('⚠️ Raw output:'); print(parsed.get('raw','')); return

    SEP = '═'*62; sep = '─'*62
    print(f'\n{SEP}\n  🩺  MEDREASON — STRUCTURED DIAGNOSTIC ANALYSIS\n{SEP}')
    print(f'\n📋  CLINICAL SUMMARY\n{sep}')
    print(parsed.get('summary','N/A'))
    print(f'\n🚨  RED FLAGS\n{sep}')
    for f in parsed.get('risk_flags',[]): print(f'  • {f}')
    print(f'\n🔍  RANKED DIFFERENTIALS\n{sep}')
    em = {'High':'🔴','Medium':'🟡','Low':'🟢'}
    for dx in parsed.get('differential_ranked',[]):
        lk = dx.get('likelihood','')
        print(f"  #{dx.get('rank','?')} {em.get(lk,'⚪')} [{lk}]  {dx.get('diagnosis','')}")
        print(f"      ↳ {dx.get('justification','')}\n")
    print(f'🧪  NEXT DIAGNOSTIC STEPS\n{sep}')
    for t in parsed.get('next_tests',[]): print(f'  • {t}')
    conf = parsed.get('confidence_level','N/A')
    c_em = {'High':'✅','Medium':'⚠️','Low':'❓'}.get(conf,'')
    print(f'\n{c_em}  CONFIDENCE: {conf}\n{sep}')
    print(parsed.get('confidence_rationale',''))
    print(f'\n{SEP}\n  ⚠️  EDUCATIONAL USE ONLY — NOT CLINICAL ADVICE\n{SEP}')


def analyze(age, sex, symptoms, history, vitals='', labs=''):
    print('⏳ Sending case to MedGemma...')
    parsed = parse_output(query_medgemma(build_case(age, sex, symptoms, history, vitals, labs)))
    display(parsed)
    return parsed

print('✅ MedReason engine ready.')

✅ MedReason engine ready.


In [4]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="google/medgemma-4b-it",
    token=os.environ["HFTOKEN"],
    device_map="auto"
)
print("✅ MedGemma loaded")

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

✅ MedGemma loaded


In [14]:

def query_medgemma(case_prompt):
    prompt = f"""{SYSTEM_PROMPT}

USER_CASE:
{case_prompt}
"""

    output = pipe(
    prompt,
    max_new_tokens=700,
    temperature=0.3,
    top_p=0.9,
    do_sample=True,
    return_full_text=False
)

    return output[0]["generated_text"]

In [6]:
FAST_MAX_NEW_TOKENS = 80

def query_medgemma(case_prompt):
    prompt = f"""{SYSTEM_PROMPT}

IMPORTANT:
- Output MUST be valid JSON only.
- Keep each field short.
- Provide at most 3 differentials and at most 3 next tests.

USER_CASE:
{case_prompt}
"""
    output = pipe(
        prompt,
        max_new_tokens=FAST_MAX_NEW_TOKENS,
        temperature=0.1,
        top_p=0.9,
        do_sample=False,
        return_full_text=False
    )
    return output[0]["generated_text"]

print("✅ Updated query_medgemma for CPU-safe fast inference.")

✅ Updated query_medgemma for CPU-safe fast inference.


---
## 🧪 Case 1: Acute Chest Pain
A high-risk cardiac presentation.

In [15]:
result1 = analyze(
    age      = '58',
    sex      = 'Male',
    symptoms = 'Sudden crushing chest pain radiating to the left arm, diaphoresis, nausea. Pain 9/10, onset 45 minutes ago.',
    history  = 'Type 2 diabetes, hypertension, 20 pack-year smoking history, father had MI at age 60.',
    vitals   = 'BP 160/95 mmHg, HR 102 bpm, RR 18, SpO2 96%, Temp 37.1°C',
    labs     = 'Troponin I: pending; ECG ordered'
)

Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⏳ Sending case to MedGemma...

══════════════════════════════════════════════════════════════
  🩺  MEDREASON — STRUCTURED DIAGNOSTIC ANALYSIS
══════════════════════════════════════════════════════════════

📋  CLINICAL SUMMARY
──────────────────────────────────────────────────────────────
A 58-year-old male presents with acute onset of crushing chest pain radiating to the left arm, diaphoresis, and nausea, highly suggestive of acute coronary syndrome (ACS). His risk factors include diabetes, hypertension, smoking history, and family history of MI. The elevated blood pressure and heart rate further support this suspicion. Troponin and ECG are pending to confirm the diagnosis.

🚨  RED FLAGS
──────────────────────────────────────────────────────────────
  • ACS
  • Cardiogenic Shock

🔍  RANKED DIFFERENTIALS
──────────────────────────────────────────────────────────────
  #1 🔴 [High]  Acute Myocardial Infarction (MI)
      ↳ The patient's symptoms (crushing chest pain radiating to the left 

In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Pipeline model device:", pipe.model.device)

CUDA available: True
GPU: Tesla T4
Pipeline model device: cuda:0


In [ ]:
FAST_MAX_NEW_TOKENS = 180

def query_medgemma(case_prompt):
    prompt = f"""{SYSTEM_PROMPT}

USER_CASE:
{case_prompt}
"""
    output = pipe(
        prompt,
        max_new_tokens=FAST_MAX_NEW_TOKENS,
        temperature=0.2,
        top_p=0.9,
        do_sample=True,
        return_full_text=False
    )
    return output[0]["generated_text"]

print("✅ Updated query_medgemma to fast demo settings.")

✅ Updated query_medgemma to fast demo settings.


---
## 🧪 Case 2: Fatigue & Weight Changes
An ambiguous endocrine presentation.

In [16]:
result2 = analyze(
    age      = '24',
    sex      = 'Female',
    symptoms = 'Progressive fatigue over 6 months, unexplained weight gain of 8kg, cold intolerance, dry skin, constipation, irregular menstrual cycles.',
    history  = 'No significant past medical history. Maternal aunt has an autoimmune thyroid condition.',
    vitals   = 'BP 105/68 mmHg, HR 58 bpm, Temp 36.1°C',
    labs     = 'CBC: normal; metabolic panel: pending'
)

Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⏳ Sending case to MedGemma...
⚠️ Raw output:
```json
{
  "summary": "A 24-year-old female presents with progressive fatigue, weight gain, cold intolerance, dry skin, constipation, and irregular menstrual cycles over 6 months. Her maternal aunt has an autoimmune thyroid condition. Initial labs are pending.",
  "risk_flags": ["hypothyroidism", "autoimmune disease"],
  "differential_ranked": [
    {"rank": 1, "diagnosis": "Hypothyroidism", "justification": "The patient's symptoms (fatigue, weight gain, cold intolerance, dry skin, constipation, irregular menses) are highly suggestive of hypothyroidism. The family history of an autoimmune thyroid condition further supports this possibility.", "likelihood": "High"},
    {"rank": 2, "diagnosis": "Depression", "justification": "Depression can present with fatigue, weight changes, and other symptoms that overlap with hypothyroidism. It's important to consider this, especially given the gradual onset of symptoms.", "likelihood": "Medium"},
    {

---
## 🧪 Case 3: Pediatric Fever & Rash
An urgent pediatric presentation.

In [17]:
result3 = analyze(
    age      = '7',
    sex      = 'Male',
    symptoms = 'Fever 39.8°C for 3 days, non-blanching petechial rash on trunk and legs, neck stiffness, photophobia, severe headache.',
    history  = 'No significant history. Not up to date on meningococcal vaccine. School reported a similar illness in a classmate last week.',
    vitals   = 'BP 88/55 mmHg, HR 138 bpm, RR 24, SpO2 97%',
    labs     = 'WBC: 18,500 (elevated), CRP: markedly elevated'
)

Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⏳ Sending case to MedGemma...
⚠️ Raw output:
```json
{
  "summary": "A 7-year-old male presents with fever, petechial rash, neck stiffness, photophobia, and severe headache, concerning for a possible meningococcal infection. His vital signs show hypotension and tachycardia, suggesting possible sepsis. The elevated WBC and CRP further support an infectious process. The recent classmate's illness raises suspicion for a contagious etiology.",
  "risk_flags": ["sepsis", "meningococcemia", "meningitis"],
  "differential_ranked": [
    {"rank": 1, "diagnosis": "Meningococcemia/Meningitis", "justification": "The combination of fever, petechial rash, neck stiffness, photophobia, headache, elevated WBC, and elevated CRP strongly suggests meningococcemia or meningitis. The recent classmate's illness further supports this diagnosis.", "likelihood": "High"},
    {"rank": 2, "diagnosis": "Sepsis", "justification": "The patient's hypotension, tachycardia, and elevated WBC count are concerning for se

---
## ✏️ Try Your Own Case

In [18]:
my_result = analyze(
    age      = '67',        # e.g. '45'
    sex      = 'Male',        # e.g. 'Female'
    symptoms = 'Sudden Onset curshing chest pain, radiating to left arm, shortness of breath, nausea, sweating for 45 minutues',        # Describe presenting complaints
    history  = 'Hypertension, Type 2 Diabetes, 40 pack per yr smoking hsitory',        # Past medical, family, social history
    vitals   = 'BP 160/102, HR 112, Temp 98.6 F, SPo2 94% on room air',        # Optional: BP, HR, Temp, SpO2, etc.
    labs     = 'Troponin pending, glucose 212 mg per dl'         # Optional: any available lab results
)

Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⏳ Sending case to MedGemma...
⚠️ Raw output:
```json
{
  "summary": "A 67-year-old male presents with acute onset chest pain radiating to the left arm, shortness of breath, nausea, and diaphoresis, concerning for an acute coronary syndrome (ACS). He has significant risk factors including hypertension, type 2 diabetes, and a 40 pack-year smoking history. His vital signs show elevated blood pressure and heart rate, and decreased oxygen saturation. Troponin levels are pending.",
  "risk_flags": ["Acute Coronary Syndrome (ACS)", "Hypoxia"],
  "differential_ranked": [
    {"rank": 1, "diagnosis": "Acute Coronary Syndrome (ACS)", "justification": "The patient's symptoms (chest pain radiating to the left arm, shortness of breath, nausea, diaphoresis) are highly suggestive of ACS. His risk factors (hypertension, diabetes, smoking history) further increase the likelihood of ACS.", "likelihood": "High"},
    {"rank": 2, "diagnosis": "Pulmonary Embolism (PE)", "justification": "Shortness of breat